# P7-1 · Bài 9 · ViT và CLIP

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torch.nn as nn

class ViT(nn.Module):
    def __init__(self, p=2, d=64, depth=4, heads=4, img=8, n_cls=10):
        super().__init__()
        self.patch = nn.Conv2d(1, d, p, stride=p)                       # patch embedding
        n = (img // p) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * .02)         # embedding vị trí
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_cls))
    def forward(self, x):
        t = self.patch(x).flatten(2).transpose(1, 2)                     # (B, n, d)
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], 1) + self.pos
        return self.head(self.enc(t)[:, 0])                              # lấy đầu ra của CLS

vit = ViT()
x = torch.randn(5, 1, 8, 8)                                   # 5 ảnh xám 8×8
print(vit.patch(x).flatten(2).transpose(1, 2).shape)          # 16 mảnh 2×2, mỗi mảnh 64 chiều
print(vit(x).shape, sum(p.numel() for p in vit.parameters()), "tham số")

torch.Size([5, 16, 64])
torch.Size([5, 10]) 136138 tham số


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
# emb_train, emb_test: embedding ảnh từ encoder CLIP đã đóng băng, chuẩn hóa độ dài 1.
# Chạy thử ở đây: thay CLIP bằng một "encoder đóng băng" đơn giản (PCA) trên chữ số viết tay
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
enc = PCA(32).fit(X_tr)
norm = lambda E: E / np.linalg.norm(E, axis=1, keepdims=True)
emb_train, emb_test = norm(enc.transform(X_tr)), norm(enc.transform(X_te))

clf = LogisticRegression(C=1.0, max_iter=2000).fit(emb_train, y_train)
print("accuracy:", clf.score(emb_test, y_test))

accuracy: 0.9407407407407408


## Bài tập 4: Zero-shot với transformers {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng: tải CLIP từ Hugging Face và ảnh của bạn, chạy trên Colab
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
TEMPLATES = ["a photo of a {}.", "a product photo of a {}.", "a close-up photo of a {}."]

@torch.no_grad()
def class_embeddings(classes):
    out = []
    for c in classes:
        t = proc(text=[tpl.format(c) for tpl in TEMPLATES], return_tensors="pt", padding=True)
        e = model.get_text_features(**t)
        e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.mean(0) / e.mean(0).norm())
    return torch.stack(out)

@torch.no_grad()
def zero_shot(image_path, classes):
    img = model.get_image_features(**proc(images=Image.open(image_path).convert("RGB"), return_tensors="pt"))
    img = img / img.norm(dim=-1, keepdim=True)
    probs = (100 * img @ class_embeddings(classes).T).softmax(-1)[0]
    return dict(zip(classes, probs.tolist()))

## Bài tập 5: Patch embedding hai cách {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
img = torch.randn(1, 3, 32, 32)
P, D = 8, 48
conv = nn.Conv2d(3, D, kernel_size=P, stride=P)
pos = torch.randn(1, 16, D)                          # position embedding của lưới 4×4
# TODO: patches (1, 16, 192) bằng reshape/permute; out_a = patches @ W.T + b (W lấy từ conv); out_b từ conv; pos6 (1, 36, D)

In [ ]:
patches = img.reshape(1, 3, 4, P, 4, P).permute(0, 2, 4, 1, 3, 5).reshape(1, 16, 3 * P * P)
W = conv.weight.reshape(D, -1)
out_a = patches @ W.T + conv.bias
out_b = conv(img).flatten(2).transpose(1, 2)
grid = pos.transpose(1, 2).reshape(1, D, 4, 4)
pos6 = F.interpolate(grid, size=(6, 6), mode="bicubic", align_corners=False).flatten(2).transpose(1, 2)
print(patches.shape, out_b.shape, "giống:", torch.allclose(out_a, out_b, atol=1e-5), "| pos mới:", pos6.shape)

torch.Size([1, 16, 192]) torch.Size([1, 16, 48]) giống: True | pos mới: torch.Size([1, 36, 48])


## Bài tập 6: ViT tí hon trên chữ số viết tay {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
d = load_digits()
X = torch.tensor(d.images / 16, dtype=torch.float32).unsqueeze(1); y = torch.tensor(d.target)
X_tr, y_tr, X_va, y_va = X[:1437], y[:1437], X[1437:], y[1437:]

class TinyViT(nn.Module):
    def __init__(self, D=32):
        super().__init__()
        # TODO: patch (Conv2d 1→D, kernel 2, stride 2), cls, pos (1, 17, D), encoder, head
    def forward(self, x):
        # TODO
        return x

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, D=32):
        super().__init__()
        self.patch = nn.Conv2d(1, D, 2, stride=2)
        self.cls = nn.Parameter(torch.zeros(1, 1, D))
        self.pos = nn.Parameter(torch.randn(1, 17, D) * 0.02)
        layer = nn.TransformerEncoderLayer(D, 4, dim_feedforward=64, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(D, 10)
    def forward(self, x):
        p = self.patch(x).flatten(2).transpose(1, 2)
        h = torch.cat([self.cls.expand(len(x), -1, -1), p], 1) + self.pos
        return self.head(self.encoder(h)[:, 0])

torch.manual_seed(0)
vit = TinyViT()
opt = torch.optim.AdamW(vit.parameters(), lr=3e-3, weight_decay=0.05)
for epoch in range(30):
    vit.train()
    for i in torch.randperm(1437).split(64):
        opt.zero_grad(); nn.functional.cross_entropy(vit(X_tr[i]), y_tr[i]).backward(); opt.step()
vit.eval()
with torch.no_grad():
    acc = (vit(X_va).argmax(1) == y_va).float().mean().item()
print(f"ViT: {sum(p.numel() for p in vit.parameters())} tham số, val acc {acc:.3f}")

ViT: 18154 tham số, val acc 0.919


## Bài tập 7: Zero-shot với nhiều mẫu câu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
D, lop = 64, ["bàn trang điểm", "gương", "ghế đôn"]
tam = rng.normal(size=(3, D))
anh = np.repeat(tam, 100, 0) + rng.normal(0, 2.2, (300, D)); nhan = np.repeat(np.arange(3), 100)
mau_cau = tam[None] + rng.normal(0, 3.0, (4, 3, D))     # 4 mẫu câu × 3 lớp, mỗi mẫu câu lệch một kiểu
norm = lambda v: v / np.linalg.norm(v, axis=-1, keepdims=True)
anh, mau_cau = norm(anh), norm(mau_cau)
# TODO: acc_tung_mau (list 4 số), acc_ensemble

In [ ]:
acc_tung_mau = [float(((anh @ mau_cau[k].T).argmax(1) == nhan).mean()) for k in range(4)]
tb = norm(mau_cau.mean(0))
acc_ensemble = float(((anh @ tb.T).argmax(1) == nhan).mean())
print("từng mẫu câu:", [round(a, 3) for a in acc_tung_mau], "| trung bình 4 mẫu:", round(acc_ensemble, 3))

từng mẫu câu: [0.747, 0.663, 0.657, 0.68] | trung bình 4 mẫu: 0.877
